# 03 · Full KV-cache baseline over the dataset
Runs `scripts/run_experiments.py` (model loaded once, one record per sample saved immediately to
`results/raw_logs/<experiment_id>.jsonl`, JSON + CSV rebuilt after each sample). If the kernel dies, re-run the
same cell with `--resume`. Make sure notebook 02's kernel is shut down.

In [ ]:
# Run from anywhere: find the project root and make it importable
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
PY = sys.executable
print("project root:", ROOT)

In [ ]:
# Hugging Face token: kept in this kernel's environment only, never written to disk or printed.
# Qwen2.5-Coder is not gated, so the token is optional, but it avoids anonymous rate limits.
import os, getpass
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (Enter to skip): ").strip()
    if not os.environ["HF_TOKEN"]:
        del os.environ["HF_TOKEN"]
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
# Put the model cache on the persistent volume of the space
os.environ.setdefault("HF_HOME", str(Path.home() / "hf_cache"))
print("HF_HOME:", os.environ["HF_HOME"])

In [ ]:
import torch
free, total = torch.cuda.mem_get_info()
print(f"free GPU memory: {free/1024**3:.1f} of {total/1024**3:.1f} GiB")
assert free / total > 0.8, "Another process is using the GPU (notebook 02 still running?). Shut it down first."

## 1. Quick check: 3 samples (~1–2 min)

In [ ]:
!{PY} scripts/run_experiments.py --max-samples 3 --experiment-id smoke_3 --overwrite

## 2. Full baseline: 30 samples, 8K budget, 256 new tokens (~10–20 min on L4)
For a long run you can also start it from a terminal so it survives a closed browser tab:
`nohup python scripts/run_experiments.py > results/raw_logs/baseline_001.log 2>&1 &`

In [ ]:
!{PY} scripts/run_experiments.py --experiment-id baseline_001 --resume

In [ ]:
import pandas as pd
df = pd.read_csv("results/csv/baseline_001.csv")
cols = ["sample_id", "workflow", "status", "input_tokens", "output_tokens", "total_inference_seconds", "ttft_seconds",
        "tpot_ms", "kv_cache_bytes", "peak_allocated_gpu_bytes", "syntax_valid", "functional_correctness"]
df[[c for c in cols if c in df]]

## 3. Optional: same tasks padded to ~8K tokens (task code in the middle)
Still the full KV cache. This is the long-context baseline that later compression methods are compared against.

In [ ]:
!{PY} scripts/run_experiments.py --config configs/experiment_longctx.yaml --resume